<a href="https://colab.research.google.com/github/lincoln-bot/Assignment-2-HASTS206-GROUP-6/blob/main/assignment2_demographic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd

# If running in Google Colab and the local 'data/' folder isn't present yet, clone the repo
if not os.path.exists("data"):
    !git clone https://github.com/lincoln-bot/Assignment-2-HASTS206-GROUP-6.git
    %cd Assignment-2-HASTS206-GROUP-6

# Load directly using clean relative paths (works in Colab, VS Code, and Jupyter)
districts = pd.read_csv("data/districts.csv")
age_sex = pd.read_csv("data/age_sex.csv")
life_table = pd.read_csv("data/life_table_inputs.csv")
scenarios = pd.read_csv("data/projection_scenarios.csv")
insured = pd.read_csv("data/insured_transitions.csv")

with open("data/README.txt", "r") as f:
    print(f.read())

HASTS206 | Assignment 2 | Dataset guide

All records are synthetic and describe hypothetical districts and insured lives
in a Zimbabwean teaching case. Dates and district labels are illustrative.
Extract this ZIP and keep the original CSV files for reproducibility.
Missing values are blank. Counts represent people unless stated otherwise.

districts.csv (24 rows, key district_id)
  population_2020 and population_2026 are opening stocks on 1 January.
  The opening 2025 stock is obtained by summing age_sex.csv.
  The four event counts cover calendar 2025 and close the balance at 1 Jan 2026.
  piped_water_pct is a district household percentage. Two values are missing.
  region is a synthetic reporting label.

age_sex.csv (864 rows, key district_id + age_start + sex)
  age_start gives 5-year bands (0 = 0-4, 5 = 5-9, ..., 85 = 85+).
  sex is F or M. population_2025 is the 1 Jan 2025 count.
  baseline_deaths_2025 is for residents present at baseline, classified by age
  at baseline. District

In [5]:
# ==============================================================================
# SETUP & IMPORTS (Relative Paths + GitHub Auto-Sync for Google Colab)
# ==============================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import expm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

print("All Libraries imported successfully!")

# If running in Google Colab and 'data/' isn't present yet, clone the GitHub repo
if not os.path.exists("data"):
    !git clone https://github.com/lincoln-bot/Assignment-2-HASTS206-GROUP-6.git
    %cd Assignment-2-HASTS206-GROUP-6

os.makedirs("outputs", exist_ok=True)

# Load all 5 datasets using clean relative paths
districts = pd.read_csv("data/districts.csv")
age_sex = pd.read_csv("data/age_sex.csv")
life_table = pd.read_csv("data/life_table_inputs.csv")
scenarios = pd.read_csv("data/projection_scenarios.csv")
insured = pd.read_csv("data/insured_transitions.csv")

with open("data/README.txt", "r") as f:
    print("\n=== DATASET GUIDE (README.txt) ===")
    print(f.read())

print("All five datasets loaded successfully!")

# ==============================================================================
# BEFORE ANALYSIS: DATA-QUALITY LOG, KEY CHECKS & BALANCING RECONCILIATION
# ==============================================================================
tables = {
    "districts": (districts, ["district_id"]),
    "age_sex": (age_sex, ["district_id", "age_start", "sex"]),
    "life_table_inputs": (life_table, ["age_start"]),
    "projection_scenarios": (scenarios, ["scenario"]),
    "insured_transitions": (insured, ["district_id", "origin", "destination"]),
}

dq_rows = []
for name, (df_tbl, keys) in tables.items():
    dup_keys = int(df_tbl.duplicated(subset=keys).sum())
    null_cols = df_tbl.isnull().sum()
    missing_summary = ", ".join([f"{c}:{int(v)}" for c, v in null_cols.items() if v > 0]) or "None"
    dq_rows.append({
        "Table": name,
        "Rows": len(df_tbl),
        "Columns": df_tbl.shape[1],
        "Primary_Key": " + ".join(keys),
        "Duplicate_Keys": dup_keys,
        "Missing_Values": missing_summary
    })

dq_log = pd.DataFrame(dq_rows)
print("\n=== DATA QUALITY LOG ===")
print(dq_log.to_string(index=False))

# Verify age-band and sex coverage across all 24 districts
print("\nUnique age_start bands:", sorted(age_sex["age_start"].unique()))
print("Unique sex values:", sorted(age_sex["sex"].unique()))
print("Districts with full 36 age-sex cells:", (age_sex.groupby("district_id").size() == 36).sum(), "/ 24")

# Reconcile 2025 opening stock & 2026 balancing equation across all 24 districts
agg_2025 = age_sex.groupby("district_id", as_index=False).agg(
    pop_2025_sum=("population_2025", "sum"),
    baseline_deaths_sum=("baseline_deaths_2025", "sum"),
    births_female_sum=("births_to_women_2025", "sum"),
    arrivals_sum=("arrivals_2025", "sum"),
    departures_sum=("departures_2025", "sum"),
)

recon = districts.merge(agg_2025, on="district_id", validate="1:1")
recon["reconstructed_pop_2026"] = (
    recon["pop_2025_sum"]
    + recon["births_2025"]
    - recon["deaths_2025"]
    + recon["arrivals_2025"]
    - recon["departures_2025"]
)
recon["pop_2026_diff"] = recon["population_2026"] - recon["reconstructed_pop_2026"]
recon["births_diff"] = recon["births_2025"] - recon["births_female_sum"]
recon["deaths_scope_diff"] = recon["deaths_2025"] - recon["baseline_deaths_sum"]

print("\nMax 2026 population balance discrepancy:", recon["pop_2026_diff"].abs().max())
print("Max births reconciliation discrepancy:", recon["births_diff"].abs().max())
print("\nDeaths scope difference (district deaths_2025 minus age_sex baseline_deaths_2025):")
print(recon[["district_id", "deaths_2025", "baseline_deaths_sum", "deaths_scope_diff"]].head(8).to_string(index=False))

# Export reconciled district baseline table for downstream sections & Power BI
recon.to_csv("outputs/reconciled_districts.csv", index=False)
print("\nSaved outputs/reconciled_districts.csv successfully.")

All Libraries imported successfully!

=== DATASET GUIDE (README.txt) ===
HASTS206 | Assignment 2 | Dataset guide

All records are synthetic and describe hypothetical districts and insured lives
in a Zimbabwean teaching case. Dates and district labels are illustrative.
Extract this ZIP and keep the original CSV files for reproducibility.
Missing values are blank. Counts represent people unless stated otherwise.

districts.csv (24 rows, key district_id)
  population_2020 and population_2026 are opening stocks on 1 January.
  The opening 2025 stock is obtained by summing age_sex.csv.
  The four event counts cover calendar 2025 and close the balance at 1 Jan 2026.
  piped_water_pct is a district household percentage. Two values are missing.
  region is a synthetic reporting label.

age_sex.csv (864 rows, key district_id + age_start + sex)
  age_start gives 5-year bands (0 = 0-4, 5 = 5-9, ..., 85 = 85+).
  sex is F or M. population_2025 is the 1 Jan 2025 count.
  baseline_deaths_2025 is for

In [6]:
import os, glob
import pandas as pd

folder = "assignment_data"
files = glob.glob(folder + "/*.csv")
data = {}
log = []

# 1. Load original datasets and inspect them
for file in files:
    name = os.path.basename(file)
    df = pd.read_csv(file)
    data[name] = df

    log.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Numeric Columns": sum(
            pd.api.types.is_numeric_dtype(df[c]) for c in df.columns
        )
    })

    print(f"\n{name}")
    print("Columns:", list(df.columns))
    print("Data types:\n", df.dtypes)

# 2. Save data-quality log
quality_log = pd.DataFrame(log)
os.makedirs("outputs", exist_ok=True)
quality_log.to_csv("outputs/before_analysis_quality_log.csv", index=False)

# 3. Check possible identifier columns
for name, df in data.items():
    for col in df.columns:
        if col.lower() in ["district_id", "district", "id"]:
            print(name, col, "Unique:", df[col].nunique(),
                  "Duplicates:", df[col].duplicated().sum())

# 4. Check numerical ranges
for name, df in data.items():
    print(f"\nNumerical ranges: {name}")
    print(df.select_dtypes(include="number").describe())

print("\nBEFORE ANALYSIS CHECKS COMPLETED.")
print("Quality log saved in outputs folder.")


BEFORE ANALYSIS CHECKS COMPLETED.
Quality log saved in outputs folder.
